# EDA — Twitch Game Pulse

Análisis exploratorio de audiencia de videojuegos en Twitch.

**Datos reales** capturados mediante snapshots diarios desde el 27/08/2026 hasta el 02/09/2026.

**Fuente de datos:** Twitch Helix API → `data/twitch_pulse.db`

In [ ]:
import sqlite3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import timedelta
import os

DB_PATH = '../data/twitch_pulse.db'

if not os.path.exists(DB_PATH):
    raise FileNotFoundError(f'No se encontró {DB_PATH}. Ejecuta primero: python ingesta.py')

## 1. Carga y vista general

In [ ]:
conn = sqlite3.connect(DB_PATH)

df = pd.read_sql('''
    SELECT s.id, s.juego_id, j.nombre, s.timestamp, s.viewers, s.num_streams
    FROM snapshots_audiencia s
    JOIN juegos j ON s.juego_id = j.id
    ORDER BY s.timestamp
''', conn)

df['timestamp'] = pd.to_datetime(df['timestamp'], format='ISO8601', utc=True)
df['fecha'] = df['timestamp'].dt.date

NO_JUEGOS = ['Just Chatting', 'IRL', 'Slots', 'Sports', 'Music', 'Art', 'ASMR', 'Crypto',
             'Talk Shows & Podcasts', 'DJs', 'Poker', 'Animals, Aquariums, and Zoos',
             'Co-working & Studying', 'Pools, Hot Tubs, and Beaches', 'Politics',
             'Software and Game Development', 'Special Events', 'Science & Technology',
             'Games + Demos', 'Retro', 'Education']
df = df[~df['nombre'].isin(NO_JUEGOS)]

print(f'Shape: {df.shape}')
print(f'Rango de fechas: {df.fecha.min()} a {df.fecha.max()}')
print(f'Juegos únicos: {df.nombre.nunique()}')
print(f'Total de snapshots: {len(df)}')
df.head()

### Conclusión

El dataset contiene 1.156 registros de 198 videojuegos capturados entre el 13/08/2026 y el 02/09/2026. Cada registro representa un snapshot de audiencia con viewers, streams y timestamp.

## 2. Ranking de audiencia — ¿Quién gana la batalla por la atención?

In [ ]:
idx_ultimo = df.groupby('nombre')['timestamp'].idxmax()
ultimo = df.loc[idx_ultimo].sort_values('viewers', ascending=False)

top10 = ultimo.head(10)
print('Top 10 juegos por viewers (último snapshot):\n')
for i, (_, row) in enumerate(top10.iterrows(), 1):
    print(f'{i:2d}. {row["nombre"]:30s} | {row["viewers"]:>10,} viewers | {row["num_streams"]:>5} streams')

# Concentración de audiencia
top5_viewers = ultimo.head(5)['viewers'].sum()
total_viewers = ultimo['viewers'].sum()
print(f'\nTop 5 concentran {top5_viewers:,} viewers ({top5_viewers/total_viewers*100:.1f}% del total)')

fig = px.bar(
    top10,
    x='nombre',
    y='viewers',
    color='viewers',
    color_continuous_scale='viridis',
    title='Top 10 juegos por viewers (último snapshot)',
    labels={'nombre': 'Juego', 'viewers': 'Viewers'},
)
fig.update_layout(xaxis_tickangle=-45, showlegend=False)
fig.show()

### Conclusión

En el último snapshot, **GTA V** lidera con **129.510 viewers**, seguido de **Minecraft** (121.375) y **League of Legends** (110.127). Los 5 primeros concentran solo el **27.0%** de la audiencia total, lo que indica una distribución relativamente equilibrada entre los grandes títulos. La atención no está uniformemente distribuida, pero tampoco concentrada en un solo juego.

## 3. Evolución temporal — Top 5 juegos

In [ ]:
top5_nombres = ultimo.head(5)['nombre'].tolist()
df_top5 = df[df['nombre'].isin(top5_nombres)]

diario = df_top5.groupby(['fecha', 'nombre'])['viewers'].mean().reset_index()
diario['fecha'] = pd.to_datetime(diario['fecha'])

fig = px.line(
    diario,
    x='fecha',
    y='viewers',
    color='nombre',
    title='Evolución diaria de audiencia — Top 5',
    labels={'fecha': 'Fecha', 'viewers': 'Viewers (media diaria)', 'nombre': 'Juego'},
    markers=True,
)
fig.update_layout(legend_title_text='Juego')
fig.show()

print('Estadísticas por juego:')
for nombre in top5_nombres:
    sub = diario[diario['nombre']==nombre]
    print(f'  {nombre}: min={sub["viewers"].min():,.0f}, max={sub["viewers"].max():,.0f}, media={sub["viewers"].mean():,.0f}')

### Conclusión

La forma de la línea es más importante que la posición. Counter-Strike muestra una audiencia media alta y relativamente estable (min=76.986, max=239.662). League of Legends presenta picos pronunciados (hasta 225.300 viewers). World of Warcraft muestra una subida drástica en los últimos días (de 4.381 a 89.937 viewers). **La posición de un juego y su comportamiento son dos cosas diferentes.**

## 4. Crecimiento — ¿Quién está ganando audiencia?

In [ ]:
ahora = df['timestamp'].max()
inicio_actual = ahora - timedelta(days=7)
inicio_anterior = inicio_actual - timedelta(days=7)

semana_actual = df[(df['timestamp'] > inicio_actual) & (df['timestamp'] <= ahora)]
semana_anterior = df[(df['timestamp'] > inicio_anterior) & (df['timestamp'] <= inicio_actual)]

agg_actual = semana_actual.groupby('nombre')['viewers'].mean().reset_index()
agg_actual.columns = ['nombre', 'viewers_actual']
agg_anterior = semana_anterior.groupby('nombre')['viewers'].mean().reset_index()
agg_anterior.columns = ['nombre', 'viewers_anterior']

merged = agg_actual.merge(agg_anterior, on='nombre', how='outer').fillna(0)
merged['crecimiento_pct'] = merged.apply(
    lambda r: ((r['viewers_actual'] - r['viewers_anterior']) / r['viewers_anterior'] * 100)
    if r['viewers_anterior'] > 0 else 0, axis=1
)

top_crec = merged[merged['viewers_anterior'] > 100].sort_values('crecimiento_pct', ascending=False).head(10)
print('Top 10 crecimiento (semana actual vs anterior):\n')
for i, (_, r) in enumerate(top_crec.iterrows(), 1):
    print(f'{i:2d}. {r["nombre"]:30s} | {r["viewers_anterior"]:>8,.0f} → {r["viewers_actual"]:>8,.0f} | {r["crecimiento_pct"]:+6.1f}%')

fig = px.bar(
    top_crec.sort_values('crecimiento_pct', ascending=True),
    x='crecimiento_pct',
    y='nombre',
    orientation='h',
    color='crecimiento_pct',
    color_continuous_scale='RdYlGn',
    title='Top 10 crecimiento de audiencia (%)',
    labels={'nombre': 'Juego', 'crecimiento_pct': 'Crecimiento (%)'},
)
fig.update_layout(showlegend=False, yaxis=dict(autorange='reversed'))
fig.show()

### Conclusión

Los juegos con mayor crecimiento son **World of Warcraft (+1.077%)**, **Deadlock (+843%)** y **Delta Force (+632%)**. Counter-Strike, aunque no lidera el ranking por volumen, muestra crecimiento positivo. El ranking me dice quién está arriba, el crecimiento me dice hacia dónde se mueve la atención.

## 5. Variabilidad — ¿Todos se comportan igual?

In [ ]:
stats = df.groupby('nombre')['viewers'].agg(['mean', 'std', 'max', 'min', 'count']).reset_index()
stats.columns = ['nombre', 'media', 'desv_est', 'maximo', 'minimo', 'count']
stats['cv'] = stats['desv_est'] / stats['media'] * 100
stats = stats[stats['count'] >= 5].sort_values('cv', ascending=False)

top_var = stats.head(10)
print('Top 10 por variabilidad (CV %):\n')
for i, (_, r) in enumerate(top_var.iterrows(), 1):
    print(f'{i:2d}. {r["nombre"]:30s} | CV: {r["cv"]:6.1f}% | Media: {r["media"]:>10,.0f} | Max: {r["maximo"]:>10,}')

fig = px.scatter(
    stats,
    x='media',
    y='cv',
    size='maximo',
    hover_name='nombre',
    title='Variabilidad vs Audiencia media (tamaño = viewers máximos)',
    labels={'media': 'Audiencia media', 'cv': 'Coeficiente de variación (%)'},
)
fig.show()

### Conclusión

Los juegos con mayor variabilidad son **Rust (CV: 140.9%)**, **Rainbow Six Siege (133.2%)** y **World of Warcraft (131.2%)**. Un CV alto indica que la audiencia fluctúa mucho: puede ser impulsada por eventos puntuales. Delta Force (CV: 113.2%) destaca por ser menos masivo pero con comportamiento inestable. **Una única métrica puede ocultar información.**

## 6. Eficiencia — Viewers por stream

In [ ]:
ultimo_vps = ultimo.copy()
ultimo_vps['viewers_per_stream'] = ultimo_vps['viewers'] / ultimo_vps['num_streams']
top_vps = ultimo_vps.sort_values('viewers_per_stream', ascending=False).head(10)

print('Top 10 viewers por stream (último snapshot):\n')
for i, (_, r) in enumerate(top_vps.iterrows(), 1):
    print(f'{i:2d}. {r["nombre"]:30s} | {r["viewers"]:>8,} viewers | {r["num_streams"]:>5} streams | {r["viewers_per_stream"]:>6.0f} v/s')

fig = px.bar(
    top_vps.sort_values('viewers_per_stream', ascending=True),
    x='viewers_per_stream',
    y='nombre',
    orientation='h',
    color='viewers_per_stream',
    color_continuous_scale='plasma',
    title='Top 10 viewers por stream (eficiencia)',
    labels={'nombre': 'Juego', 'viewers_per_stream': 'Viewers / Stream'},
)
fig.update_layout(showlegend=False)
fig.show()

### Conclusión

Juegos como **Poppy Playtime** (5.779 v/s) y **BALL BOY Simulator** (12.303 v/s) tienen pocos streams pero alta audiencia por stream, lo que indica que unos pocos creadores concentran toda la atención. En contraste, **VALORANT** tiene 3.650 streams pero solo 20 v/s, distribuyendo la audiencia entre muchos creadores.

## 7. Síntesis — Las tres perspectivas

Para entender qué ocurre en Twitch no basta con preguntar «¿quién tiene más viewers?». Necesitamos combinar:

| Perspectiva | Qué mide | Ejemplo |
|---|---|---|
| **Volumen** | Cuánta audiencia tiene un juego | GTA V: 129.510 viewers |
| **Evolución** | Si está creciendo o decreciendo | WoW: +1.077% vs semana anterior |
| **Variabilidad** | Si su comportamiento es estable | Rust: CV 140.9% (muy inestable) |

Un juego puede liderar el ranking pero ser estable (Counter-Strike). Otro puede ser menor pero estar en crecimiento explosivo (WoW). Y otro puede tener audiencia moderada pero comportamiento impredecible (Delta Force).